# UA-DETRAC Offline Video-Based SORT Tracking (Modular Edition)

Notebook này chạy thuật toán **SORT (Simple Online and Realtime Tracking)** bằng cách đọc trực tiếp các **file detections (.txt) đã được YOLO sinh ra trước đó** (lưu trong `./all_predictions`), kết hợp đọc hình ảnh từ **file video (.mp4) gốc** để xuất ra video đã được vẽ tracking và ID hoàn chỉnh.

## 📂 Ưu điểm của phương pháp Modular Offline:
- **Không cần chạy lại YOLO**: Tiết kiệm 100% tài nguyên GPU. YOLO chỉ cần chạy một lần duy nhất ở notebook trước để lưu nhãn detections dạng file text.
- **Tách biệt và Gọn nhẹ**: Notebook này chạy cực kỳ nhẹ trên CPU, không cần import hay load mô hình PyTorch/Ultralytics nặng nề vào bộ nhớ.
- **Đọc trực tiếp Video (.mp4)**: Dùng `cv2.VideoCapture` để lấy hình ảnh trực tiếp từ video gốc làm nền vẽ tracking, không cần giải nén hàng ngàn ảnh lẻ tẻ.

## 📤 Đầu ra kép (Double Output):
1. **Video kết quả đã tracking** (`.mp4`): Video gốc đã được vẽ đè lên các bounding box xanh lá và ID di chuyển của từng xe mượt mà.
2. **File text kết quả tracking** (MOT format): Ghi lại lịch sử tracking của từng sequence dưới dạng 1 file text duy nhất lưu ở `./sort_predictions`, sẵn sàng đưa vào `metric.ipynb` để tính toán chỉ số MOTA.

In [ ]:
# Tự động cài đặt filterpy nếu chưa có trong môi trường
try:
    import filterpy

    print("✓ Thư viện filterpy đã được cài đặt sẵn!")
except ImportError:
    print("⚠️ Không tìm thấy filterpy. Đang tiến hành cài đặt tự động...")
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "filterpy"])
    print("✓ Cài đặt filterpy thành công!")

import os
import re
import sys
from glob import glob
from pathlib import Path
from typing import List, Tuple, Dict
import xml.etree.ElementTree as ET

import cv2
import numpy as np
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

print("✓ Khởi tạo môi trường SORT thành công!")


## 1. Các hàm trợ giúp và Xử lý Ignored Regions

In [ ]:
def find_annotation_xml(sequence_name: str, annotations_folder: str = None) -> str:
    """Tìm file XML ground-truth chứa thông tin Ignored Region."""
    if not sequence_name or not annotations_folder:
        return None
    candidates = [
        os.path.join(annotations_folder, f"{sequence_name}.xml"),
        os.path.join(annotations_folder, sequence_name, f"{sequence_name}.xml"),
    ]
    for path in candidates:
        if os.path.isfile(path):
            return os.path.abspath(path)
    return None


def load_detrac_ignored_regions_xml(xml_path: str) -> List[np.ndarray]:
    """Đọc các hộp Ignored Region từ file XML."""
    ignored_regions = []
    if not xml_path or not os.path.isfile(xml_path):
        return ignored_regions

    root = ET.parse(xml_path).getroot()
    ignored_node = root.find("ignored_region")
    if ignored_node is None:
        return ignored_regions

    for box_node in ignored_node.findall("box"):
        left = float(box_node.attrib["left"])
        top = float(box_node.attrib["top"])
        width = float(box_node.attrib["width"])
        height = float(box_node.attrib["height"])
        ignored_regions.append(
            np.array([left, top, left + width, top + height], dtype=float)
        )

    return ignored_regions


def bbox_area(bbox: np.ndarray) -> float:
    return max(0.0, float(bbox[2] - bbox[0])) * max(0.0, float(bbox[3] - bbox[1]))


def intersection_area(box_a: np.ndarray, box_b: np.ndarray) -> float:
    x1 = max(float(box_a[0]), float(box_b[0]))
    y1 = max(float(box_a[1]), float(box_b[1]))
    x2 = min(float(box_a[2]), float(box_b[2]))
    y2 = min(float(box_a[3]), float(box_b[3]))
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def ignored_overlap_ratio(bbox: np.ndarray, ignored_regions: List[np.ndarray]) -> float:
    area = bbox_area(bbox)
    if area <= 0.0 or not ignored_regions:
        return 0.0
    ignored_area = sum(intersection_area(bbox, region) for region in ignored_regions)
    return min(1.0, ignored_area / area)


def filter_detections_by_ignore(
    detections: np.ndarray, ignored_regions: List[np.ndarray], overlap_threshold: float
) -> np.ndarray:
    if detections.size == 0 or not ignored_regions:
        return detections
    keep = [
        ignored_overlap_ratio(det[:4], ignored_regions) < overlap_threshold
        for det in detections
    ]
    return detections[np.array(keep, dtype=bool)] if keep else detections


def filter_tracks_by_ignore(
    tracks: List[tuple[int, np.ndarray]],
    ignored_regions: List[np.ndarray],
    overlap_threshold: float,
):
    if not tracks or not ignored_regions:
        return tracks
    return [
        (track_id, bbox)
        for track_id, bbox in tracks
        if ignored_overlap_ratio(bbox, ignored_regions) < overlap_threshold
    ]


def iou(bb_test: np.ndarray, bb_gt: np.ndarray) -> float:
    """Tính Intersection over Union (IoU) giữa 2 hộp giới hạn."""
    xx1 = np.maximum(bb_test[0], bb_gt[0])
    yy1 = np.maximum(bb_test[1], bb_gt[1])
    xx2 = np.minimum(bb_test[2], bb_gt[2])
    yy2 = np.minimum(bb_test[3], bb_gt[3])

    w = np.maximum(0.0, xx2 - xx1)
    h = np.maximum(0.0, yy2 - yy1)
    inter = w * h

    area1 = (bb_test[2] - bb_test[0]) * (bb_test[3] - bb_test[1])
    area2 = (bb_gt[2] - bb_gt[0]) * (bb_gt[3] - bb_gt[1])
    union = area1 + area2 - inter

    return inter / union if union > 0.0 else 0.0


def read_yolo_label(
    txt_path: str, img_w: int, img_h: int, conf_threshold: float = 0.0
) -> np.ndarray:
    """Đọc YOLO label (.txt) đã được lưu và chuyển đổi bbox chuẩn hóa sang tọa độ pixel."""
    detections = []
    if not os.path.exists(txt_path):
        return np.zeros((0, 5), dtype=float)

    with open(txt_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 6:
                continue

            _, xc, yc, w, h, conf = map(float, parts[:6])
            if conf < conf_threshold:
                continue

            xc *= img_w
            yc *= img_h
            w *= img_w
            h *= img_h

            x1 = xc - w / 2.0
            y1 = yc - h / 2.0
            x2 = xc + w / 2.0
            y2 = yc + h / 2.0

            detections.append([x1, y1, x2, y2, conf])

    return (
        np.array(detections, dtype=float)
        if detections
        else np.zeros((0, 5), dtype=float)
    )


def visualize_ignored_regions(
    frame: np.ndarray, ignored_regions: List[np.ndarray], alpha: float = 0.25
) -> np.ndarray:
    """Vẽ các vùng bị bỏ qua màu đỏ trong suốt lên frame hình."""
    if not ignored_regions:
        return frame
    overlay = frame.copy()
    for region in ignored_regions:
        x1, y1, x2, y2 = map(int, region.tolist())
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 255), -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2, lineType=cv2.LINE_AA)
    cv2.addWeighted(overlay, alpha, frame, 1.0 - alpha, 0, dst=frame)
    return frame


def visualize_tracks(
    frame: np.ndarray, tracks: List[tuple[int, np.ndarray]]
) -> np.ndarray:
    """Vẽ bounding box màu xanh lá và ID lên các xe."""
    for track_id, bbox in tracks:
        x1, y1, x2, y2 = map(int, bbox.tolist())
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2, lineType=cv2.LINE_AA)
        cv2.putText(
            frame,
            f"ID {track_id}",
            (x1, max(0, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
            lineType=cv2.LINE_AA,
        )
    return frame


## 2. Thiết lập Lớp Track & Sort

In [ ]:
def convert_bbox_to_z(bbox: np.ndarray) -> np.ndarray:
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    cx = bbox[0] + w / 2.0
    cy = bbox[1] + h / 2.0
    s = w * h
    r = w / float(h + 1e-6)
    return np.array([cx, cy, s, r], dtype=float).reshape((4, 1))


def convert_x_to_bbox(x: np.ndarray) -> np.ndarray:
    cx, cy, s, r = x[0, 0], x[1, 0], x[2, 0], x[3, 0]
    s = max(s, 1e-6)
    r = max(r, 1e-6)
    w = np.sqrt(s * r)
    h = np.sqrt(s / r)
    x1 = cx - w / 2.0
    y1 = cy - h / 2.0
    x2 = cx + w / 2.0
    y2 = cy + h / 2.0
    return np.array([x1, y1, x2, y2], dtype=float).reshape((1, 4))


class Track:
    count = 0

    def __init__(self, bbox: np.ndarray) -> None:
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.id = Track.count
        Track.count += 1

        self.time_since_update = 0
        self.hits = 1
        self.hit_streak = 1
        self.age = 0
        self.bbox = bbox.copy()

        self.kf.F = np.eye(7, dtype=float)
        self.kf.F[0, 4] = 1.0
        self.kf.F[1, 5] = 1.0
        self.kf.F[2, 6] = 1.0

        self.kf.H = np.zeros((4, 7), dtype=float)
        self.kf.H[0, 0] = 1.0
        self.kf.H[1, 1] = 1.0
        self.kf.H[2, 2] = 1.0
        self.kf.H[3, 3] = 1.0

        self.kf.R *= 0.01
        self.kf.Q *= 0.01
        self.kf.P *= 10.0
        self.kf.x[:4] = convert_bbox_to_z(bbox)

    def predict(self) -> np.ndarray:
        self.kf.predict()
        self.age += 1
        self.time_since_update += 1
        self.bbox = convert_x_to_bbox(self.kf.x)[0]
        return self.bbox

    def update(self, bbox: np.ndarray) -> None:
        self.time_since_update = 0
        self.hits += 1
        self.hit_streak += 1
        self.bbox = bbox.copy()
        self.kf.update(convert_bbox_to_z(bbox))

    def get_state(self) -> np.ndarray:
        return self.bbox


class Sort:
    def __init__(
        self,
        max_age: int = 3,
        min_hits: int = 1,
        iou_threshold: float = 0.3,
        center_distance_threshold: float = 1.4,
    ) -> None:
        self.max_age = max_age
        self.min_hits = min_hits
        self.iou_threshold = iou_threshold
        self.center_distance_threshold = center_distance_threshold
        self.trackers: List[Track] = []

    def _associate_detections_to_trackers(
        self, detections: np.ndarray, trackers: np.ndarray
    ) -> tuple[List[tuple[int, int]], List[int], List[int]]:
        if len(trackers) == 0:
            return [], list(range(len(detections))), []

        iou_matrix = np.zeros((len(detections), len(trackers)), dtype=float)
        distance_matrix = np.zeros((len(detections), len(trackers)), dtype=float)
        score_matrix = np.zeros((len(detections), len(trackers)), dtype=float)

        for d, det in enumerate(detections):
            for t, trk in enumerate(trackers):
                iou_matrix[d, t] = iou(det, trk)
                det_center = np.array(
                    [(det[0] + det[2]) / 2.0, (det[1] + det[3]) / 2.0]
                )
                trk_center = np.array(
                    [(trk[0] + trk[2]) / 2.0, (trk[1] + trk[3]) / 2.0]
                )
                det_diag = np.linalg.norm([det[2] - det[0], det[3] - det[1]])
                trk_diag = np.linalg.norm([trk[2] - trk[0], trk[3] - trk[1]])
                norm = max(1.0, 0.5 * (det_diag + trk_diag))
                distance_matrix[d, t] = np.linalg.norm(det_center - trk_center) / norm
                distance_score = max(
                    0.0, 1.0 - distance_matrix[d, t] / self.center_distance_threshold
                )
                score_matrix[d, t] = iou_matrix[d, t] + distance_score

        row_ind, col_ind = linear_sum_assignment(-score_matrix)
        matches = []
        unmatched_detections = list(range(len(detections)))
        unmatched_trackers = list(range(len(trackers)))

        for d, t in zip(row_ind, col_ind):
            if (
                iou_matrix[d, t] < self.iou_threshold
                and distance_matrix[d, t] > self.center_distance_threshold
            ):
                continue
            matches.append((d, t))
            unmatched_detections.remove(d)
            unmatched_trackers.remove(t)

        return matches, unmatched_detections, unmatched_trackers

    def update(self, detections: np.ndarray) -> List[tuple[int, np.ndarray]]:
        outputs: List[tuple[int, np.ndarray]] = []

        if detections.size == 0:
            for tracker in self.trackers:
                tracker.predict()
                tracker.hit_streak = 0
            self.trackers = [
                tracker
                for tracker in self.trackers
                if tracker.time_since_update <= self.max_age
            ]
            return outputs

        if len(self.trackers) == 0:
            for det in detections:
                self.trackers.append(Track(det))
            for tracker in self.trackers:
                if tracker.hits >= self.min_hits:
                    outputs.append((tracker.id, tracker.get_state()))
            return outputs

        predicted_boxes = np.zeros((len(self.trackers), 4), dtype=float)
        for t, tracker in enumerate(self.trackers):
            predicted_boxes[t] = tracker.predict()

        matches, unmatched_dets, unmatched_trks = (
            self._associate_detections_to_trackers(detections, predicted_boxes)
        )

        for det_idx, trk_idx in matches:
            self.trackers[trk_idx].update(detections[det_idx])

        for det_idx in unmatched_dets:
            self.trackers.append(Track(detections[det_idx]))

        for trk_idx in unmatched_trks:
            self.trackers[trk_idx].hit_streak = 0

        self.trackers = [
            tracker
            for tracker in self.trackers
            if tracker.time_since_update <= self.max_age
        ]

        for tracker in self.trackers:
            if tracker.hits >= self.min_hits and tracker.time_since_update == 0:
                outputs.append((tracker.id, tracker.get_state()))

        return outputs


## 3. Cấu hình Đường dẫn và Tham số trên Kaggle

In [ ]:
# === CẤU HÌNH ĐƯỜNG DẪN KAGGLE ===

# 1. Thư mục chứa các file .txt detections YOLO đã chạy trước đó
PREDICTIONS_FOLDER = "/kaggle/working/all_predictions"
if not os.path.exists(PREDICTIONS_FOLDER):
    PREDICTIONS_FOLDER = "./all_predictions"  # Fallback local

# 2. Thư mục chứa các file video .mp4 tập Test gốc (Dùng làm nền vẽ tracking)
VIDEOS_FOLDER = "/kaggle/input/datasets/bratjay/ua-detrac-orig/ua_detrac_test_videos"
if not os.path.exists(VIDEOS_FOLDER):
    VIDEOS_FOLDER = "./ua_detrac_test_videos"  # Fallback local

# 3. Thư mục chứa XML Annotations Test (Để lọc Ignored Regions)
ANNOTATIONS_FOLDER = "/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML"

# 4. Thư mục đầu ra lưu VIDEO đã được SORT tracking thành công
OUTPUT_VIDEOS_FOLDER = "/kaggle/working/tracked_videos"

# 5. Thư mục đầu ra lưu FILE TEXT kết quả tracking (chuẩn định dạng MOT để đánh giá metrics)
OUTPUT_TRACKS_FOLDER = "/kaggle/working/sort_predictions"

# --- THAM SỐ PHÂN TÍCH & SORT ---
LABEL_FRAME_STRIDE = 4  # Nếu yolo được cấu hình chạy nhảy cóc mỗi 4 frame, chỉnh 4
MAX_AGE = LABEL_FRAME_STRIDE + 2
MIN_HITS = 1
IOU_THRESHOLD_SORT = 0.2
CENTER_DISTANCE_THRESHOLD = 1.4
CONF_THRESHOLD = 0.30  # Lọc lại độ tự tin nếu cần

# --- THAM SỐ IGNORE ---
FILTER_IGNORED_REGIONS = True
IGNORE_OVERLAP_THRESHOLD = 0.50
IGNORE_REGION_ALPHA = 0.25

# --- TỰ ĐỘNG PHÁT HIỆN ĐƯỜNG DẪN TRÊN KAGGLE (ĐỀ PHÒNG ĐƯỜNG DẪN KHÁC) ---
kaggle_input = Path("/kaggle/input")
if kaggle_input.is_dir():
    # Tự động tìm kiếm thư mục videos
    v_dir = list(kaggle_input.rglob("ua_detrac_test_videos"))
    if v_dir:
        VIDEOS_FOLDER = str(v_dir[-1])

    # Tự động tìm kiếm XML Annotations
    ann_dir = list(kaggle_input.rglob("DETRAC-Test-Annotations-XML"))
    if ann_dir:
        ANNOTATIONS_FOLDER = str(ann_dir[-1])

print("✓ Cấu hình hoàn tất!")
print(f"Detections Folder: {PREDICTIONS_FOLDER}")
print(f"Videos Folder: {VIDEOS_FOLDER}")
print(f"XML Annotations: {ANNOTATIONS_FOLDER}")
print(f"Output Tracks: {OUTPUT_TRACKS_FOLDER}")
print(f"Output Videos: {OUTPUT_VIDEOS_FOLDER}")


## 4. Thực thi Đọc Detections + Đọc Video và Chạy SORT Tracking

In [ ]:
from pathlib import Path

out_tracks_dir = Path(OUTPUT_TRACKS_FOLDER)
out_videos_dir = Path(OUTPUT_VIDEOS_FOLDER)

out_tracks_dir.mkdir(parents=True, exist_ok=True)
out_videos_dir.mkdir(parents=True, exist_ok=True)

if not os.path.exists(PREDICTIONS_FOLDER):
    raise FileNotFoundError(
        f"❌ Không tìm thấy thư mục nhãn detections tại: {PREDICTIONS_FOLDER}"
    )
if not os.path.isdir(VIDEOS_FOLDER):
    raise FileNotFoundError(f"❌ Không tìm thấy thư mục video tại: {VIDEOS_FOLDER}")

# Quét các file video gốc .mp4 đầu vào
video_paths = sorted(list(Path(VIDEOS_FOLDER).glob("*.mp4")))
print(f"[*] Tìm thấy {len(video_paths)} videos để thực thi vẽ tracking.")

if len(video_paths) == 0:
    print("❌ Không tìm thấy file .mp4 video nào để chạy!")
else:
    # Duyệt và chạy từng video
    for idx, v_path in enumerate(video_paths):
        seq_name = v_path.stem  # Ví dụ: MVI_39031
        print(f"\n➔ [{idx+1}/{len(video_paths)}] Đang xử lý video: {v_path.name}")

        # Reset ID cho mỗi video mới
        Track.count = 0
        tracker = Sort(
            max_age=MAX_AGE,
            min_hits=MIN_HITS,
            iou_threshold=IOU_THRESHOLD_SORT,
            center_distance_threshold=CENTER_DISTANCE_THRESHOLD,
        )

        # Tải Ignored Regions của sequence tương ứng từ XML
        xml_path = find_annotation_xml(seq_name, ANNOTATIONS_FOLDER)
        ignored_regions = load_detrac_ignored_regions_xml(xml_path) if xml_path else []
        if ignored_regions:
            print(
                f"   [i] Lọc {len(ignored_regions)} ignored regions dựa theo file XML."
            )

        # Mở video đầu vào để làm nền hình ảnh vẽ tracking
        cap = cv2.VideoCapture(str(v_path))
        if not cap.isOpened():
            print(f"   ❌ Lỗi: Không thể mở video {v_path.name}")
            continue

        # Lấy thông số video
        v_fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        print(
            f"   [i] FPS: {v_fps:.2f} | Kích thước: {width}x{height} | Tổng số frame: {total_frames}"
        )

        # Khởi tạo VideoWriter để lưu video kết quả đã vẽ tracking
        output_video_path = out_videos_dir / f"{seq_name}_tracked.mp4"
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out_video = cv2.VideoWriter(
            str(output_video_path), fourcc, v_fps, (width, height)
        )

        if not out_video.isOpened():
            print(
                f"   ❌ Lỗi: Không thể khởi tạo VideoWriter tại {output_video_path.name}"
            )
            cap.release()
            continue

        # Chuẩn bị file kết quả tracking dạng .txt (cho metric.ipynb)
        output_txt_path = out_tracks_dir / f"{seq_name}.txt"
        txt_lines = []

        frame_idx = 0

        with tqdm(total=total_frames, desc=f"      {seq_name}", leave=False) as pbar:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                frame_idx += 1
                pbar.update(1)

                # Tìm file label .txt đã được lưu của frame này
                label_txt_path = os.path.join(
                    PREDICTIONS_FOLDER, f"{seq_name}_img{frame_idx:05d}.txt"
                )

                # 1. Đọc detections đã lưu trên đĩa từ file text
                detections = read_yolo_label(
                    label_txt_path, width, height, CONF_THRESHOLD
                )

                # 2. Lọc các bounding box trùng với Ignored Region của XML
                if FILTER_IGNORED_REGIONS and ignored_regions:
                    detections = filter_detections_by_ignore(
                        detections, ignored_regions, IGNORE_OVERLAP_THRESHOLD
                    )

                bboxes = (
                    detections[:, :4]
                    if detections.size
                    else np.zeros((0, 4), dtype=float)
                )

                # 3. Cập nhật tracker SORT để liên kết ID các xe
                tracks = tracker.update(bboxes)

                if FILTER_IGNORED_REGIONS and ignored_regions:
                    tracks = filter_tracks_by_ignore(
                        tracks, ignored_regions, IGNORE_OVERLAP_THRESHOLD
                    )

                # 4. Ghi kết quả vào danh sách xuất file text (MOT định dạng)
                for track_id, bbox in tracks:
                    x1, y1, x2, y2 = bbox
                    left = x1
                    top = y1
                    width_bbox = x2 - x1
                    height_bbox = y2 - y1

                    line = f"{frame_idx},{track_id},{left:.2f},{top:.2f},{width_bbox:.2f},{height_bbox:.2f},-1,-1,-1,-1\n"
                    txt_lines.append(line)

                # 5. Vẽ trực quan kết quả (Ignored Regions màu đỏ + Tracks màu xanh lá) lên frame ảnh
                vis_frame = frame.copy()
                if FILTER_IGNORED_REGIONS and ignored_regions:
                    vis_frame = visualize_ignored_regions(
                        vis_frame, ignored_regions, IGNORE_REGION_ALPHA
                    )
                vis_frame = visualize_tracks(vis_frame, tracks)

                # Ghi frame đã vẽ trực quan vào VideoWriter
                out_video.write(vis_frame)

        # Giải phóng tài nguyên của video hiện tại
        cap.release()
        out_video.release()

        # Ghi toàn bộ kết quả tracking của sequence ra file text
        with open(output_txt_path, "w", encoding="utf-8") as out_f:
            out_f.writelines(txt_lines)

        print(f"   ✓ Đã xuất video đã vẽ tracking: {output_video_path.name}")
        print(
            f"   ✓ Đã xuất file text kết quả tracking: {output_txt_path.name} ({len(txt_lines)} dòng)"
        )

    print("\n" + "=" * 80)
    print(f"🎉 Hoàn thành! Đã chạy xong SORT cho toàn bộ các file video.")
    print(f"📂 Video kết quả (.mp4) lưu tại: {out_videos_dir.resolve()}")
    print(f"📂 File text MOT (.txt) lưu tại: {out_tracks_dir.resolve()}")
    print("=" * 80)


## 5. Nén kết quả để tải về (Tùy chọn)

In [ ]:
import shutil

# Nén toàn bộ video đã được tracking
zip_videos = "/kaggle/working/tracked_videos_archive"
if os.path.exists(OUTPUT_VIDEOS_FOLDER) and len(os.listdir(OUTPUT_VIDEOS_FOLDER)) > 0:
    print(f"📦 Đang nén thư mục video đã tracking {OUTPUT_VIDEOS_FOLDER}...")
    shutil.make_archive(zip_videos, "zip", OUTPUT_VIDEOS_FOLDER)
    print(f"🎉 Đã nén thành công video! File lưu tại: {zip_videos}.zip")

# Nén file text tracking để chạy evaluation
zip_tracks = "/kaggle/working/sort_predictions_archive"
if os.path.exists(OUTPUT_TRACKS_FOLDER) and len(os.listdir(OUTPUT_TRACKS_FOLDER)) > 0:
    print(f"📦 Đang nén thư mục nhãn text tracking {OUTPUT_TRACKS_FOLDER}...")
    shutil.make_archive(zip_tracks, "zip", OUTPUT_TRACKS_FOLDER)
    print(f"🎉 Đã nén thành công tracks! File lưu tại: {zip_tracks}.zip")
